# Smart India Real Estate Analytics — Data Validation

**Notebook 02: Data Validation**

This notebook validates the raw real estate dataset before any EDA, preprocessing, or modeling. The goal is to determine whether the observed data is sufficiently reliable and internally consistent to proceed with exploratory analysis.

- Dataset path: `../data/raw/Real Estate Data V21.csv`
- Scope: schema, data types, semantic missing values, malformed values, duplicates, numeric ranges, and logical consistency.
- Raw data immutability: the dataset is loaded into `df_raw` and is not modified.
- Excluded: preprocessing, feature engineering, train/test splitting, model training, and any permanent changes to the raw CSV.

In [20]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.options.display.max_colwidth = 180


## Load and Verify Dataset

Load the raw CSV into a clearly named DataFrame and verify the file path, row count, column count, and column names.

In [21]:
DATA_PATH = Path('../data/raw/Real Estate Data V21.csv')
print(f'Dataset path: {DATA_PATH}')
print(f'Path exists: {DATA_PATH.exists()}')

df_raw = pd.read_csv(DATA_PATH)
print('Dataset loaded successfully.')
print('Shape:', df_raw.shape)
print('Columns:')
print(df_raw.columns.tolist())

Dataset path: ..\data\raw\Real Estate Data V21.csv
Path exists: True
Dataset loaded successfully.
Shape: (14528, 9)
Columns:
['Name', 'Property Title', 'Price', 'Location', 'Total_Area', 'Price_per_SQFT', 'Description', 'Baths', 'Balcony']


## Schema Validation

Compare the actual columns with the expected raw schema and report missing or unexpected columns, as well as order differences.

In [22]:
expected_columns = [
    'Name',
    'Property Title',
    'Price',
    'Location',
    'Total_Area',
    'Price_per_SQFT',
    'Description',
    'Baths',
    'Balcony',
]

actual_columns = df_raw.columns.tolist()
missing_columns = [col for col in expected_columns if col not in actual_columns]
unexpected_columns = [col for col in actual_columns if col not in expected_columns]
order_matches = actual_columns == expected_columns

validation_schema = pd.DataFrame([
    {'Check': 'Expected columns present', 'Result': bool(not missing_columns), 'Details': missing_columns or 'None'},
    {'Check': 'Unexpected columns present', 'Result': bool(unexpected_columns), 'Details': unexpected_columns or 'None'},
    {'Check': 'Column order matches expected schema', 'Result': order_matches, 'Details': 'Yes' if order_matches else 'No'},
])
validation_schema

,Check,Result,Details
0,Expected columns present,True,None
1,Unexpected columns present,False,None
2,Column order matches expected schema,True,Yes


## Data-Type Validation

Inspect actual dtypes, non-null counts, and unique counts for each column. Pay special attention to Price and the numeric-looking fields.

In [23]:
dtype_summary = pd.DataFrame({
    'column': df_raw.columns,
    'dtype': df_raw.dtypes.astype(str).values,
    'non-null count': df_raw.notna().sum().values,
    'unique count': [df_raw[col].nunique(dropna=False) for col in df_raw.columns],
})
dtype_summary

,column,dtype,non-null count,unique count
0,Name,str,14528,9998
1,Property Title,str,14528,6507
2,Price,str,14528,891
3,Location,str,14528,7050
4,Total_Area,int64,14528,1774
5,Price_per_SQFT,float64,14528,2094
6,Description,str,14528,14490
7,Baths,int64,14528,6
8,Balcony,str,14528,2


## Semantic Missing-Value Validation

Identify string-based values that look like missing data even though pandas does not recognize them as null. This includes empty strings, whitespace-only strings, and common missing-value tokens.

In [24]:
semantic_missing_tokens = {
    "",
    "n/a",
    "na",
    "nan",
    "null",
    "none",
    "unknown",
    "not available",
    "-"
}

semantic_missing_summary = []

string_columns = df_raw.select_dtypes(include=["object", "string"]).columns

for column in string_columns:
    values = df_raw[column].astype("string").str.strip().str.lower()
    mask = values.isin(semantic_missing_tokens)

    semantic_missing_summary.append({
        "Column": column,
        "Semantic Missing Count": int(mask.sum())
    })

semantic_missing_summary = pd.DataFrame(semantic_missing_summary)

semantic_missing_summary

,Column,Semantic Missing Count
0,Name,0
1,Property Title,0
2,Price,0
3,Location,0
4,Description,0
5,Balcony,0


## Price Validation

The `Price` column is inspected for its observed formats, units, ambiguous values, and values that cannot be confidently interpreted.

The raw `Price` column is treated as immutable. A separate validation-only numeric representation may be created when required for consistency checks and leakage analysis.

The validation will determine:

- the price formats and units present in the dataset
- whether price values can be consistently interpreted
- whether any values are ambiguous or require further investigation
- whether a temporary numeric representation can be created without modifying the raw data

No target values are changed or artificially corrected in this notebook.

In [25]:
price_raw = df_raw['Price'].astype(str)

def classify_price_format(value):
    if pd.isna(value):
        return 'missing'
    text = str(value).strip().replace(' ', ' ')
    if text == '':
        return 'empty'
    if re.search(r'\b[Ll]acs\b', text):
        return 'Lacs'
    if re.search(r'\b[Kk]\b', text):
        return 'k'
    if text.endswith(('Cr', 'CR', 'cr', 'Cr.', 'cr.')):
        return 'Cr'
    if text.endswith(('L', 'l', 'L.', 'l.')):
        return 'L'
    if text.startswith('₹') and re.search(r'[0-9]', text):
        return 'unknown format'
    return 'other'

price_format = price_raw.map(classify_price_format)
price_format_counts = price_format.value_counts(dropna=False).rename_axis('price_format').reset_index(name='count')

def parse_price_to_rupees(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().replace(' ', ' ').replace(',', '')
    if text == '':
        return np.nan
    text = text.replace('Rs', '').replace('Rs.', '').replace('₹', '').strip()
    text = text.replace(' ', '')
    if text.lower().endswith('cr'):
        multiplier = 1e7
        text = text[:-2]
    elif text.lower().endswith('lacs'):
        multiplier = 1e5
        text = text[:-4]
    elif text.lower().endswith('l'):
        multiplier = 1e5
        text = text[:-1]
    elif text.lower().endswith('k'):
        multiplier = 1e3
        text = text[:-1]
    else:
        return np.nan
    try:
        return float(text) * multiplier
    except ValueError:
        return np.nan

price_validation = price_raw.map(parse_price_to_rupees)
price_malformed = df_raw.loc[price_validation.isna(), ['Price', 'Total_Area', 'Price_per_SQFT']].head(10)
price_validation_summary = pd.DataFrame({
    'metric': ['parsed numeric count', 'parsed missing or ambiguous count'],
    'value': [int(price_validation.notna().sum()), int(price_validation.isna().sum())],
})
display(price_format_counts)
display(price_validation_summary)
display(price_malformed)


,price_format,count
0,L,10294
1,Cr,4229
2,unknown format,4
3,Lacs,1


,metric,value
0,parsed numeric count,14525
1,parsed missing or ambiguous count,3


,Price,Total_Area,Price_per_SQFT
4277,₹2.0,3000,0.0
5856,₹3.0,2800,0.0
7948,₹1.0,1800,0.0


## Total_Area Validation

Inspect the numeric distribution of `Total_Area` and identify zero, negative, or extreme values that may require investigation.

In [26]:
total_area = df_raw['Total_Area']
total_area_summary = total_area.describe().to_frame().rename(columns={'Total_Area': 'value'})
area_extremes = pd.DataFrame([
    {'metric': 'min', 'value': int(total_area.min())},
    {'metric': '25%', 'value': float(total_area.quantile(0.25))},
    {'metric': '50%', 'value': float(total_area.median())},
    {'metric': '75%', 'value': float(total_area.quantile(0.75))},
    {'metric': 'max', 'value': int(total_area.max())},
    {'metric': 'zero count', 'value': int((total_area == 0).sum())},
    {'metric': 'negative count', 'value': int((total_area < 0).sum())},
])
area_top_values = total_area.value_counts().head(10).rename_axis('Total_Area').reset_index(name='count')
total_area_summary, area_extremes, area_top_values

(              value
 count  14528.000000
 mean    1297.916988
 std     1245.694305
 min       70.000000
 25%      650.000000
 50%     1000.000000
 75%     1439.000000
 max    35000.000000,
            metric    value
 0             min     70.0
 1             25%    650.0
 2             50%   1000.0
 3             75%   1439.0
 4             max  35000.0
 5      zero count      0.0
 6  negative count      0.0,
    Total_Area  count
 0        1200    517
 1        1000    419
 2         900    330
 3         600    312
 4        1100    259
 5         450    255
 6         800    252
 7        1500    250
 8         500    244
 9         550    193)

## Price_per_SQFT Validation

Inspect `Price_per_SQFT` for unexpected numeric values, zero or negative entries, and the raw range of observed values.

In [27]:
price_per_sqft = df_raw['Price_per_SQFT']
price_per_sqft_summary = price_per_sqft.describe().to_frame().rename(columns={'Price_per_SQFT': 'value'})
price_per_sqft_extremes = pd.DataFrame([
    {'metric': 'min', 'value': float(price_per_sqft.min())},
    {'metric': '25%', 'value': float(price_per_sqft.quantile(0.25))},
    {'metric': '50%', 'value': float(price_per_sqft.median())},
    {'metric': '75%', 'value': float(price_per_sqft.quantile(0.75))},
    {'metric': 'max', 'value': float(price_per_sqft.max())},
    {'metric': 'zero count', 'value': int((price_per_sqft == 0).sum())},
    {'metric': 'negative count', 'value': int((price_per_sqft < 0).sum())},
])
price_per_sqft_top = price_per_sqft.value_counts().head(20).rename_axis('Price_per_SQFT').reset_index(name='count')
price_per_sqft_summary, price_per_sqft_extremes, price_per_sqft_top

(               value
 count   14528.000000
 mean    11719.456222
 std     49036.068632
 min         0.000000
 25%      4480.000000
 50%      6050.000000
 75%      9312.500000
 max    999000.000000,
            metric     value
 0             min       0.0
 1             25%    4480.0
 2             50%    6050.0
 3             75%    9312.5
 4             max  999000.0
 5      zero count       3.0
 6  negative count       0.0,
     Price_per_SQFT  count
 0           5000.0    233
 1          10000.0    157
 2           4000.0    152
 3           6000.0    116
 4           6670.0    109
 5           7500.0     97
 6           6250.0     78
 7           8000.0     78
 8           5550.0     78
 9           8330.0     77
 10          3330.0     74
 11          4500.0     73
 12          4170.0     67
 13          3000.0     66
 14          4440.0     65
 15          5500.0     60
 16         12500.0     60
 17          3750.0     57
 18         20000.0     56
 19          7000.0     55)

## Price_per_SQFT Leakage Investigation

Investigate whether `Price_per_SQFT` is mathematically derived from `Price / Total_Area`. Use a validation-only numeric representation of `Price` and compare it with the raw `Price_per_SQFT` values using tolerance-based matching.

In [28]:
validation_price = price_validation
leakage_df = df_raw.assign(
    price_validation=validation_price,
    implied_price_per_sqft=lambda x: x['price_validation'] / x['Total_Area'],
)
valid_rows = (leakage_df['Total_Area'] > 0) & leakage_df['Price_per_SQFT'].notna() & leakage_df['price_validation'].notna()
leakage_df = leakage_df.loc[valid_rows].copy()
leakage_df['abs_difference'] = (leakage_df['implied_price_per_sqft'] - leakage_df['Price_per_SQFT']).abs()
leakage_df['relative_difference'] = np.where(
    leakage_df['Price_per_SQFT'] == 0,
    np.inf,
    leakage_df['abs_difference'] / leakage_df['Price_per_SQFT'],
)
leakage_counts = {
    'comparable_records': int(len(leakage_df)),
    'dataset_rows': int(len(df_raw)),
    'comparable_pct': float(len(leakage_df) / len(df_raw) * 100),
    'exact_matches': int((leakage_df['abs_difference'] == 0).sum()),
    'within_0.5pct': int((leakage_df['relative_difference'] <= 0.005).sum()),
    'within_1pct': int((leakage_df['relative_difference'] <= 0.01).sum()),
    'within_2pct': int((leakage_df['relative_difference'] <= 0.02).sum()),
    'above_2pct': int((leakage_df['relative_difference'] > 0.02).sum()),
    'zero_price_per_sqft': int((leakage_df['Price_per_SQFT'] == 0).sum()),
} 
leakage_counts


{'comparable_records': 14525,
 'dataset_rows': 14528,
 'comparable_pct': 99.97935022026432,
 'exact_matches': 2599,
 'within_0.5pct': 14409,
 'within_1pct': 14413,
 'within_2pct': 14413,
 'above_2pct': 112,
 'zero_price_per_sqft': 0}

In [29]:
leakage_match_summary = pd.DataFrame([
    {'description': 'exact match', 'count': int((leakage_df['abs_difference'] == 0).sum())},
    {'description': 'within 0.5% relative difference', 'count': int((leakage_df['relative_difference'] <= 0.005).sum())},
    {'description': 'within 1% relative difference', 'count': int((leakage_df['relative_difference'] <= 0.01).sum())},
    {'description': 'within 2% relative difference', 'count': int((leakage_df['relative_difference'] <= 0.02).sum())},
    {'description': 'more than 2% relative difference', 'count': int((leakage_df['relative_difference'] > 0.02).sum())},
])
leakage_match_summary

,description,count
0,exact match,2599
1,within 0.5% relative difference,14409
2,within 1% relative difference,14413
3,within 2% relative difference,14413
4,more than 2% relative difference,112


In [30]:
leakage_examples_match = leakage_df.loc[leakage_df['relative_difference'] <= 0.02, [
    'Price', 'Total_Area', 'Price_per_SQFT', 'price_validation', 'implied_price_per_sqft', 'abs_difference', 'relative_difference'
]].head(10)
leakage_examples_nonmatch = leakage_df.loc[leakage_df['relative_difference'] > 0.02, [
    'Price', 'Total_Area', 'Price_per_SQFT', 'price_validation', 'implied_price_per_sqft', 'abs_difference', 'relative_difference'
]].head(10)
leakage_examples_match, leakage_examples_nonmatch

(      Price  Total_Area  Price_per_SQFT  price_validation  \
 0  ₹1.99 Cr        2583          7700.0        19900000.0   
 1  ₹2.25 Cr        7000          3210.0        22500000.0   
 2   ₹1.0 Cr        1320          7580.0        10000000.0   
 3  ₹3.33 Cr        4250          7840.0        33300000.0   
 4   ₹48.0 L         960          5000.0         4800000.0   
 5   ₹40.0 L         940          4250.0         4000000.0   
 6   ₹60.0 L         880          6820.0         6000000.0   
 7  ₹72.35 L        1700          4250.0         7235000.0   
 8   ₹42.0 L         840          5000.0         4200000.0   
 9   ₹30.0 L         535          5610.0         3000000.0   
 
    implied_price_per_sqft  abs_difference  relative_difference  
 0             7704.219899        4.219899             0.000548  
 1             3214.285714        4.285714             0.001335  
 2             7575.757576        4.242424             0.000560  
 3             7835.294118        4.705882          

## Leakage Conclusion

The validation strongly indicates that `Price_per_SQFT` is derived from `Price` and `Total_Area`.

Among the comparable records, approximately 99.23% of observations have a `Price_per_SQFT` value within 2% of the value reconstructed from `Price / Total_Area`. The small differences observed in many records are consistent with rounding, while a small subset of records shows larger inconsistencies.

Therefore, `Price_per_SQFT` is considered a **target-derived variable** for this project.

### Modeling Decision

`Price_per_SQFT` should **not be used as an input feature when predicting `Price`**, because it contains information derived directly from the target variable.

The raw `Price_per_SQFT` column will remain unchanged in the raw dataset. Its exclusion will be implemented later in the modeling/preprocessing pipeline rather than by modifying the raw CSV.

In [31]:
leakage_decision = {
    'comparable_records': len(leakage_df),
    'matched_within_2pct': int((leakage_df['relative_difference'] <= 0.02).sum()),
    'matched_pct_2pct': float((leakage_df['relative_difference'] <= 0.02).mean() * 100),
    'mismatch_records': int((leakage_df['relative_difference'] > 0.02).sum()),
    'mismatch_pct': float((leakage_df['relative_difference'] > 0.02).mean() * 100),
    'price_malformed_count': int(price_validation.isna().sum()),
    'zero_price_per_sqft_in_comparable': int((leakage_df['Price_per_SQFT'] == 0).sum()),
}
leakage_decision

{'comparable_records': 14525,
 'matched_within_2pct': 14413,
 'matched_pct_2pct': 99.2289156626506,
 'mismatch_records': 112,
 'mismatch_pct': 0.7710843373493975,
 'price_malformed_count': 3,
 'zero_price_per_sqft_in_comparable': 0}

## Duplicate Validation

Inspect exact duplicate records and report counts and representative duplicate groups. Do not remove duplicates in this notebook.

In [32]:
duplicate_mask = df_raw.duplicated(keep=False)
duplicate_rows = int(df_raw.duplicated().sum())
duplicate_pct = float(duplicate_rows / len(df_raw) * 100)
duplicate_groups = df_raw.loc[duplicate_mask].groupby(list(df_raw.columns)).size().reset_index(name='count')
duplicate_groups_summary = duplicate_groups.sort_values('count', ascending=False).head(10)
representative_duplicates = df_raw.loc[duplicate_mask].head(10)
{
    'duplicate_rows': duplicate_rows,
    'duplicate_pct': duplicate_pct,
    'duplicate_groups': int(duplicate_groups.shape[0]),
}, duplicate_groups_summary, representative_duplicates

({'duplicate_rows': 8,
  'duplicate_pct': 0.05506607929515419,
  'duplicate_groups': 8},
                                                               Name  \
 0                 Asian suncity ,Police Colony, Kondapur,Hyderabad   
 1                                                  Janapriya Nivas   
 2                                         Kasavanahalli, Bangalore   
 3                                                          Nandika   
 4  Premier inspira maplewood,Belur Nagasandra, Bellandur,Bangalore   
 5                                   RKH Blessings B And C Building   
 6                                                Siruniam, Chennai   
 7                        Swapnamanjil,Adipally, Santoshpur,Kolkata   
 
                                             Property Title     Price  \
 0               3 BHK Flat for sale in Kondapur, Hyderabad  ₹1.35 Cr   
 1  2 BHK Flat for sale in Thammenahalli Village, Bangalore   ₹20.0 L   
 2          2 BHK Flat for sale in Kasavanahalli, B

## Baths Validation

Inspect `Baths` for its type and distribution. Report any zero or negative values and explain whether the values are internally consistent.

In [33]:
baths = df_raw['Baths']
baths_summary = baths.describe().to_frame().rename(columns={'Baths': 'value'})
baths_counts = baths.value_counts().sort_index().rename_axis('Baths').reset_index(name='count')
baths_zero = int((baths == 0).sum())
baths_negative = int((baths < 0).sum())
{
    'dtype': str(baths.dtype),
    'unique_values': baths.unique().tolist(),
    'zero_count': baths_zero,
    'negative_count': baths_negative,
}, baths_summary, baths_counts

({'dtype': 'int64',
  'unique_values': [4, 6, 3, 5, 2, 1],
  'zero_count': 0,
  'negative_count': 0},
               value
 count  14528.000000
 mean       2.751239
 std        0.898243
 min        1.000000
 25%        2.000000
 50%        3.000000
 75%        3.000000
 max        6.000000,
    Baths  count
 0      1    973
 1      2   4244
 2      3   7523
 3      4   1166
 4      5    456
 5      6    166)

## Balcony Validation

Inspect the actual `Balcony` values for categories, capitalization, whitespace, and missing-like representations.

In [34]:
balcony_raw = df_raw['Balcony'].astype(str)
balcony_values = balcony_raw.value_counts().rename_axis('Balcony').reset_index(name='count')
balcony_stripped = balcony_raw.str.strip()
additional_check = balcony_stripped.value_counts().rename_axis('Balcony stripped').reset_index(name='count')
missing_like_mask = balcony_stripped.isin(['', 'N/A', 'NA', 'NaN', 'null', 'None', 'Unknown', 'Not Available', '-'])
missing_like_count = int(missing_like_mask.sum())
{
    'raw_unique_count': int(balcony_raw.nunique(dropna=False)),
    'missing_like_count': missing_like_count,
}, balcony_values, additional_check

({'raw_unique_count': 2, 'missing_like_count': 0},
   Balcony  count
 0     Yes   8580
 1      No   5948,
   Balcony stripped  count
 0              Yes   8580
 1               No   5948)

## Cross-Column Logical Consistency

Check simple logical consistency rules supported by the dataset. Distinguish clearly between invalid values, unusual values, and values that require further investigation.

In [35]:
logical_checks = [
    {'check': 'Total_Area <= 0', 'count': int((df_raw['Total_Area'] <= 0).sum())},
    {'check': 'Price_per_SQFT <= 0', 'count': int((df_raw['Price_per_SQFT'] <= 0).sum())},
    {'check': 'Baths < 0', 'count': int((df_raw['Baths'] < 0).sum())},
    {'check': 'Price parse failures', 'count': int(price_validation.isna().sum())},
    {'check': 'Price_per_SQFT > 100000', 'count': int((df_raw['Price_per_SQFT'] > 100000).sum())},
]
logical_checks_df = pd.DataFrame(logical_checks)
logical_checks_df

,check,count
0,Total_Area <= 0,0
1,Price_per_SQFT <= 0,3
2,Baths < 0,0
3,Price parse failures,3
4,Price_per_SQFT > 100000,91


## Validation Summary

Summarize the main validation results and recommended next steps for preprocessing.

In [36]:
validation_summary = pd.DataFrame([
    {
        'Validation Check': 'Schema',
        'Result': 'Actual schema matches expected columns and order.',
        'Status': 'Pass',
        'Recommended Action': 'Proceed to EDA after validation review.'
    },
    {
        'Validation Check': 'Data types',
        'Result': 'Raw text fields and numeric columns were confirmed. Price is stored as text.',
        'Status': 'Warning',
        'Recommended Action': 'Plan preprocessing for Price parsing before modeling.'
    },
    {
        'Validation Check': 'Semantic missing values',
        'Result': 'No semantic missing-like values detected in string columns.',
        'Status': 'Pass',
        'Recommended Action': 'No immediate action for text missing values.'
    },
    {
        'Validation Check': 'Price representation',
        'Result': 'Most values use ₹ with Cr or L and a few entries could not be confidently interpreted.',
        'Status': 'Warning',
        'Recommended Action': 'Handle ambiguous Price entries during later preprocessing.'
    },
    {
        'Validation Check': 'Total_Area',
        'Result': 'Total_Area is integer, with no zeros or negatives. Extreme values require investigation.',
        'Status': 'Pass',
        'Recommended Action': 'Investigate unusually small and large areas during EDA.'
    },
    {
        'Validation Check': 'Price_per_SQFT leakage',
        'Result': 'Approximately 99.23% of comparable records are within 2% of Price / Total_Area, strongly indicating target-derived values.',
        'Status': 'Fail',
        'Recommended Action': 'Exclude Price_per_SQFT from model inputs when predicting Price.'
    },
    {
        'Validation Check': 'Duplicates',
        'Result': '8 exact duplicate rows detected.',
        'Status': 'Warning',
        'Recommended Action': 'Investigate duplicate rows and decide on later handling during preprocessing.'
    },
    {
        'Validation Check': 'Baths',
        'Result': 'Baths values are positive integers between 1 and 6.',
        'Status': 'Pass',
        'Recommended Action': 'Proceed to EDA on bathroom distribution.'
    },
    {
        'Validation Check': 'Balcony',
        'Result': 'Balcony contains only Yes/No values with no whitespace issues.',
        'Status': 'Pass',
        'Recommended Action': 'Evaluate as a categorical feature during preprocessing.'
    },
    {
        'Validation Check': 'Logical consistency',
        'Result': 'No negative totals or bath counts, but a small number of Price and Price_per_SQFT inconsistencies remain.',
        'Status': 'Warning',
        'Recommended Action': 'Document the inconsistent records and assess their treatment during later preprocessing.'
    },
])

validation_summary

,Validation Check,Result,Status,Recommended Action
0,Schema,Actual schema matches expected columns and order.,Pass,Proceed to EDA after validation review.
1,Data types,Raw text fields and numeric columns were confirmed. Price is stored as text.,Warning,Plan preprocessing for Price parsing before modeling.
2,Semantic missing values,No semantic missing-like values detected in string columns.,Pass,No immediate action for text missing values.
3,Price representation,Most values use ₹ with Cr or L and a few entries could not be confidently interpreted.,Warning,Handle ambiguous Price entries during later preprocessing.
4,Total_Area,"Total_Area is integer, with no zeros or negatives. Extreme values require investigation.",Pass,Investigate unusually small and large areas during EDA.
5,Price_per_SQFT leakage,"Approximately 99.23% of comparable records are within 2% of Price / Total_Area, strongly indicating target-derived values.",Fail,Exclude Price_per_SQFT from model inputs when predicting Price.
6,Duplicates,8 exact duplicate rows detected.,Warning,Investigate duplicate rows and decide on later handling during preprocessing.
7,Baths,Baths values are positive integers between 1 and 6.,Pass,Proceed to EDA on bathroom distribution.
8,Balcony,Balcony contains only Yes/No values with no whitespace issues.,Pass,Evaluate as a categorical feature during preprocessing.
9,Logical consistency,"No negative totals or bath counts, but a small number of Price and Price_per_SQFT inconsistencies remain.",Warning,Document the inconsistent records and assess their treatment during later preprocessing.


## Final Conclusions

### Key Findings

- The dataset schema is consistent with the expected structure.
- `Price` is stored as text and contains 3 ambiguous entries requiring later investigation.
- `Total_Area` contains no zero or negative values; extreme values require further investigation.
- 8 exact duplicate rows were identified.
- `Baths` contains positive integer values from 1 to 6.
- `Balcony` contains only `Yes` and `No`.

### Target Leakage

`Price_per_SQFT` is strongly indicated to be derived from `Price` and `Total_Area`. Approximately 99.23% of comparable records are within 2% of the reconstructed value.

Therefore, `Price_per_SQFT` will be excluded from model inputs when predicting `Price`.

### Data Integrity

The raw dataset was not modified. No rows, columns, or target values were changed, and no preprocessing or modeling was performed.

### Next Stage

The identified data-quality issues and variable relationships will be investigated further during EDA before making preprocessing decisions.